# Inter-rater agreement analysis

Computes Krippendorff's α (interval level) on the 21 expert-annotated QA pairs of Dataset A to assess (1) baseline agreement among the 7 human annotators and (2) the impact on group agreement when the LLM-based metric is added as an 8th rater. Complements the analysis with a per-row variance diagnostic comparing the metric's distance from the human mean on rows of high vs low human agreement.

In [32]:
import pandas as pd
import numpy as np
import krippendorff
from rich import print

In [33]:
pd.set_option('display.max_colwidth', None) # Show full content of each cell

In [34]:
raw_survey_df  = pd.read_excel("../data/survey_results.xlsx")
print(raw_survey_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Columns: 111 entries, Id to Literacy/Education Assumptions
This scale measures whether the response is pitched at an appropriate level for the user. A high score means the language is accessible without being condescending; a lo20
dtypes: datetime64[ns](2), float64(2), int64(1), object(106)
memory usage: 6.2+ KB


None

In [35]:
# The survey results follow the initial dataset, so we can merge on the question text to get the corresponding answers and contexts for each question
data = pd.read_csv('../data/human_annotation_scores.csv')
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Question  21 non-null     object 
 1   Answer    21 non-null     object 
 2   TA_score  21 non-null     float64
 3   TA_label  21 non-null     bool   
 4   CN_score  21 non-null     float64
 5   CN_label  21 non-null     bool   
dtypes: bool(2), float64(2), object(2)
memory usage: 846.0+ bytes


None

In [36]:
eval_data = pd.read_csv("../data/evaluation_results.csv")
metric_scores_ta = eval_data['Tone Attunement [GEval]']
metric_scores_cn = eval_data['Cultural Neutrality [GEval]']
print(metric_scores_ta.head())

0    0.7
1    1.0
2    1.0
3    1.0
4    0.9
Name: Tone Attunement [GEval], dtype: float64

## Survey Data Normalisation

In [37]:
# We need to map the string responses to numerical values so that we can analyse the data quantitatively.
ta_map = {
    'Ignores emotion entirely': 1,
    'Completely miscalibrated (too cold or too warm)': 1,
    'Minimal acknowledgement': 2,
    'Noticeably off': 2,
    'Acknowledges but superficially': 3,
    'Acceptable but imperfect': 3,
    'Acknowledges naturally': 4,
    'Well calibrated': 4,
    'Fully appropriate acknowledgement': 5,
    "Perfectly matched to the query's register": 5
}

cn_map = {
    'Strong unjustified assumptions': 1,
    'Talks down to or far above the user': 1,
    'Noticeable assumptions': 2,
    'Noticeably misjudged': 2,
    'Minor assumptions': 3,
    'Slightly off': 3,
    'Mostly neutral': 4,
    'Mostly appropriate': 4,
    'No assumptions made': 5,
    'Appropriately pitched': 5
}

In [38]:
# Eliminate PID columns because the responses were annonymized and thus not useful for our analysis
# Also eliminate the timestamp column because we don't need it for our analysis
survey_data = raw_survey_df.drop(columns=["Start time", "Completion time", "Email", "Name", "Language", "Id"]).copy() # eliminated Id column because it was just a repeat of the index column

# Rename columns as they contain definitions of the questions that are too long to work with easily
# Since they follow a strict repeated pattern (name+\n definition+question_number), we can rename them systhematically
scale_names = ['EA', 'TC', 'Legal', 'Cultural', 'Literacy']
survey_data.columns = scale_names * 21

# Clean first
survey_data = survey_data.map(lambda x: x.strip().replace('\u2019', "'") if isinstance(x, str) else x)
survey_data = survey_data.map(lambda x: x.strip() if isinstance(x, str) else x)

# Then map to numeric
ta_cols = [0, 1]
cn_cols = [2, 3, 4]

for i in range(21):
    for j in ta_cols:
        col_idx = i * 5 + j
        survey_data.iloc[:, col_idx] = survey_data.iloc[:, col_idx].map(ta_map)
    for j in cn_cols:
        col_idx = i * 5 + j
        survey_data.iloc[:, col_idx] = survey_data.iloc[:, col_idx].map(cn_map)

In [39]:
# Sanity check
print(survey_data.info())
print(survey_data.iloc[:, :5].head())
print(survey_data.isnull().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Columns: 105 entries, EA to Literacy
dtypes: object(105)
memory usage: 5.9+ KB


None

EA TC Legal Cultural Literacy
0  4  4     5        5        4
1  4  4     5        5        4
2  5  5     4        5        5
3  5  4     5        5        4
4  4  4     5        5        4

0

In [40]:
# Normalise the data so that all annotators are on the same scale
survey_data = survey_data.apply(lambda x: (x - 1) / 4, axis=0)
print(survey_data.head())

EA    TC Legal Cultural Literacy    EA    TC Legal Cultural Literacy  \
0  0.75  0.75   1.0      1.0     0.75  0.75  0.75   1.0      1.0      0.5   
1  0.75  0.75   1.0      1.0     0.75   1.0   1.0   1.0      1.0     0.75   
2   1.0   1.0  0.75      1.0      1.0   1.0  0.75  0.75      1.0      1.0   
3   1.0  0.75   1.0      1.0     0.75  0.75  0.75  0.25      1.0      1.0   
4  0.75  0.75   1.0      1.0     0.75  0.25  0.25   1.0      1.0      1.0   

   ...    EA    TC Legal Cultural Literacy    EA    TC Legal Cultural Literacy  
0  ...  0.75  0.75   0.5      0.5     0.75  0.75  0.75   1.0      1.0     0.75  
1  ...  0.75  0.75   1.0      1.0     0.75  0.75  0.75   1.0      1.0      1.0  
2  ...   0.5   1.0   1.0      1.0      1.0  0.75   1.0   1.0      1.0      1.0  
3  ...  0.75  0.75   1.0      1.0     0.75  0.75  0.75  0.25      1.0     0.75  
4  ...  0.25   0.5   1.0      1.0     0.75   0.5  0.75   1.0      1.0     0.75  

[5 rows x 105 columns]

In [41]:
# Weighted aggregation (one-to-one: annotator x QA pair)
for i in range(21):
  ea_col = "EA_" + str(i)
  tc_col = "TC_" + str(i)
  legal_col = "Legal_" + str(i)
  cultural_col = "Cultural_" + str(i)
  literacy_col = "Literacy_" + str(i)
  survey_data[ea_col] = survey_data.iloc[:, i*5 + 0]
  survey_data[tc_col] = survey_data.iloc[:, i*5 + 1]
  survey_data[legal_col] = survey_data.iloc[:, i*5 + 2]
  survey_data[cultural_col] = survey_data.iloc[:, i*5 + 3]
  survey_data[literacy_col] = survey_data.iloc[:, i*5 + 4]
# Drop the original columns once we've created the new ones
survey_data = survey_data.drop(columns=['EA', 'TC', 'Legal', 'Cultural', 'Literacy'])

print(survey_data.head())

C:\Users\dari\AppData\Local\Temp\ipykernel_43508\1936407737.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  survey_data[literacy_col] = survey_data.iloc[:, i*5 + 4]
C:\Users\dari\AppData\Local\Temp\ipykernel_43508\1936407737.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  survey_data[ea_col] = survey_data.iloc[:, i*5 + 0]
C:\Users\dari\AppData\Local\Temp\ipykernel_43508\1936407737.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

EA_0  TC_0 Legal_0 Cultural_0 Literacy_0  EA_1  TC_1 Legal_1 Cultural_1  \
0  0.75  0.75     1.0        1.0       0.75  0.75  0.75     1.0        1.0   
1  0.75  0.75     1.0        1.0       0.75   1.0   1.0     1.0        1.0   
2   1.0   1.0    0.75        1.0        1.0   1.0  0.75    0.75        1.0   
3   1.0  0.75     1.0        1.0       0.75  0.75  0.75    0.25        1.0   
4  0.75  0.75     1.0        1.0       0.75  0.25  0.25     1.0        1.0   

  Literacy_1  ... EA_19 TC_19 Legal_19 Cultural_19 Literacy_19 EA_20 TC_20  \
0        0.5  ...  0.75  0.75      0.5         0.5        0.75  0.75  0.75   
1       0.75  ...  0.75  0.75      1.0         1.0        0.75  0.75  0.75   
2        1.0  ...   0.5   1.0      1.0         1.0         1.0  0.75   1.0   
3        1.0  ...  0.75  0.75      1.0         1.0        0.75  0.75  0.75   
4        1.0  ...  0.25   0.5      1.0         1.0        0.75   0.5  0.75   

  Legal_20 Cultural_20 Literacy_20  
0      1.0         1.0        0.75  
1      1.0         1.0         1.0  
2      1.0         1.0         1.0  
3     0.25         1.0        0.75  
4      1.0         1.0        0.75  

[5 rows x 105 columns]

In [42]:
# Now we can compute the weighted average for each question and create new columns for the aggregated scores
for i in range(21):
  ta_col = "TA_" + str(i)
  cn_col = "CN_" + str(i)
  survey_data[ta_col] = survey_data['EA_' + str(i)] * 0.5 + survey_data['TC_' + str(i)] * 0.5
  survey_data[cn_col] = survey_data['Legal_' + str(i)] * 0.6 + survey_data['Cultural_' + str(i)] * 0.3 + survey_data['Literacy_' + str(i)] * 0.1
  survey_data = survey_data.drop(columns=['EA_' + str(i), 'TC_' + str(i), 'Legal_' + str(i), 'Cultural_' + str(i), 'Literacy_' + str(i)])

C:\Users\dari\AppData\Local\Temp\ipykernel_43508\228042968.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  survey_data[ta_col] = survey_data['EA_' + str(i)] * 0.5 + survey_data['TC_' + str(i)] * 0.5
C:\Users\dari\AppData\Local\Temp\ipykernel_43508\228042968.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  survey_data[cn_col] = survey_data['Legal_' + str(i)] * 0.6 + survey_data['Cultural_' + str(i)] * 0.3 + survey_data['Literacy_' + str(i)] * 0.1
C:\Users\dari\AppData\Local\Temp\ipykernel_43508\228042968.py:5: PerformanceW

In [43]:
print(survey_data.head())

TA_0   CN_0   TA_1   CN_1  TA_2   CN_2   TA_3   CN_3   TA_4  CN_4  ...  \
0   0.75  0.975   0.75   0.95   0.5    0.9  0.625   0.95   0.75  0.95  ...   
1   0.75  0.975    1.0  0.975  0.75  0.675   0.75   0.85    0.5   1.0  ...   
2    1.0   0.85  0.875   0.85   0.5   0.55    0.5    1.0    1.0  0.55  ...   
3  0.875  0.975   0.75   0.55   0.5  0.975    0.0  0.375  0.625  0.55  ...   
4   0.75  0.975   0.25    1.0   0.5  0.975   0.25    0.7  0.125  0.85  ...   

   TA_16  CN_16  TA_17  CN_17  TA_18  CN_18  TA_19  CN_19  TA_20  CN_20  
0  0.875  0.975   0.75  0.675   0.75  0.975   0.75  0.525   0.75  0.975  
1  0.875    1.0    1.0   0.85  0.125  0.775   0.75  0.975   0.75    1.0  
2  0.625  0.975   0.75  0.975  0.375  0.725   0.75    1.0  0.875    1.0  
3    1.0   0.85  0.625  0.525    0.0    0.0   0.75  0.975   0.75  0.525  
4   0.75   0.85  0.625    0.7   0.75    0.1  0.375  0.975  0.625  0.975  

[5 rows x 42 columns]

End state: two 7×21 matrices (rows = annotators, columns = QA pair index). Every (annotator, QA pair) pair gets its own TA score and its own CN score.

## Build rater matrices for Krippendorff

In [44]:
# Extract the TA and CN columns for Krippendorff's alpha calculation into sepparate dataframes
ta_data = survey_data[[col for col in survey_data.columns if col.startswith('TA_')]]
cn_data = survey_data[[col for col in survey_data.columns if col.startswith('CN_')]]

We know that the survey data rows align with the QA pair order. However, the DeepEval scores are misaligned with the pairs.

In [45]:
# Align the metric scores to the same QA pair order as the survey data
qa_pairs = pd.read_csv("../data/qa_pairs.csv")
merged = qa_pairs.merge(eval_data, on='user_input', how='inner')
merged = merged.drop(columns=['conv_id', 'retrieval_context', 'Answer Relevancy Reason', 'Faithfulness Reason', 'Contextual Relevancy Reason',
                              'Tone Attunement [GEval] Reason', 'Cultural Neutrality [GEval] Reason', 'bot_output_y'])
merged.rename(columns={'user_input_x': 'user_input', 'bot_output_x': 'bot_output'}, inplace=True)

In [46]:
# Stack metrics as the 8th row
metric_row_ta = merged['Tone Attunement [GEval]'].values
human_matrix_ta = survey_data[[f'TA_{i}' for i in range(21)]].values
rater_matrix_ta = np.vstack([human_matrix_ta, metric_row_ta])

In [47]:
print(rater_matrix_ta)

[[0.75 0.75 0.5 0.625 0.75 0.75 0.5 0.375 0.75 0.75 0.25 0.75 0.5 0.375
  0.75 0.75 0.875 0.75 0.75 0.75 0.75]
 [0.75 1.0 0.75 0.75 0.5 0.75 0.375 0.625 0.5 0.75 0.75 0.5 0.5 0.25 0.75
  0.375 0.875 1.0 0.125 0.75 0.75]
 [1.0 0.875 0.5 0.5 1.0 0.75 0.125 0.5 0.375 0.875 0.25 0.75 0.625 0.375
  0.875 0.625 0.625 0.75 0.375 0.75 0.875]
 [0.875 0.75 0.5 0.0 0.625 0.75 0.125 0.5 0.125 0.875 0.75 0.75 0.375
  0.75 0.75 0.125 1.0 0.625 0.0 0.75 0.75]
 [0.75 0.25 0.5 0.25 0.125 0.25 0.375 0.5 0.125 0.375 0.375 0.75 0.75
  0.25 0.875 0.375 0.75 0.625 0.75 0.375 0.625]
 [0.375 0.5 0.5 0.875 0.5 0.5 0.375 0.5 0.75 0.625 0.25 0.625 0.125 0.125
  0.875 0.375 0.875 0.75 0.125 1.0 0.875]
 [0.5 0.75 1.0 1.0 1.0 0.625 0.375 0.625 0.875 0.75 0.125 0.375 0.5 0.5
  1.0 1.0 0.5 1.0 0.75 1.0 1.0]
 [1.0 1.0 1.0 0.9 0.7 0.8 0.9 0.7 0.7 1.0 0.8 0.8 0.9 0.6 0.7 0.7 0.9 1.0
  0.2 0.8 0.8]]

In [48]:
metric_row_cn = merged['Cultural Neutrality [GEval]'].values
human_matrix_cn = survey_data[[f'CN_{i}' for i in range(21)]].values
rater_matrix_cn = np.vstack([human_matrix_cn, metric_row_cn])

In [49]:
print(rater_matrix_cn)

[[0.9749999999999999 0.95 0.8999999999999999 0.95 0.95 0.95
  0.49999999999999994 0.27499999999999997 0.725 0.75 0.475 0.75 0.475
  0.9749999999999999 0.9749999999999999 0.9749999999999999
  0.9749999999999999 0.675 0.9749999999999999 0.5249999999999999
  0.9749999999999999]
 [0.9749999999999999 0.9749999999999999 0.675 0.85 0.9999999999999999
  0.9999999999999999 0.825 0.9749999999999999 0.6249999999999999
  0.9999999999999999 0.9749999999999999 0.9999999999999999
  0.5249999999999999 0.6249999999999999 0.9999999999999999 0.825
  0.9999999999999999 0.85 0.775 0.9749999999999999 0.9999999999999999]
 [0.85 0.85 0.5499999999999999 0.9999999999999999 0.5499999999999999
  0.32499999999999996 0.75 0.675 0.45 0.6249999999999999
  0.5999999999999999 0.9999999999999999 0.9999999999999999 0.7
  0.9999999999999999 0.9999999999999999 0.9749999999999999
  0.9749999999999999 0.725 0.9999999999999999 0.9999999999999999]
 [0.9749999999999999 0.5499999999999999 0.9749999999999999 0.375
  0.5499999999999999 0.9749999999999999 0.9749999999999999
  0.5499999999999999 0.225 0.9749999999999999 0.9999999999999999
  0.8999999999999999 0.325 0.5249999999999999 0.9999999999999999
  0.9249999999999999 0.85 0.5249999999999999 0.0 0.9749999999999999
  0.5249999999999999]
 [0.9749999999999999 0.9999999999999999 0.9749999999999999 0.7 0.85 0.825
  0.9999999999999999 0.9999999999999999 0.375 0.825 0.49999999999999994
  0.9999999999999999 0.9999999999999999 0.9749999999999999 0.85 0.85 0.85
  0.7 0.1 0.9749999999999999 0.9749999999999999]
 [0.5499999999999999 0.9749999999999999 0.65 0.8 0.5249999999999999 0.575
  0.45 0.875 0.5499999999999999 0.9749999999999999 0.5999999999999999
  0.9999999999999999 0.375 0.5499999999999999 0.9249999999999999
  0.9999999999999999 0.9749999999999999 0.9999999999999999 0.6 0.85
  0.9999999999999999]
 [0.9749999999999999 0.9749999999999999 0.5499999999999999
  0.9999999999999999 0.9999999999999999 0.4 0.9999999999999999
  0.9999999999999999 0.9999999999999999 0.9999999999999999
  0.9999999999999999 0.9999999999999999 0.9999999999999999
  0.9999999999999999 0.9999999999999999 0.9999999999999999
  0.9999999999999999 0.7 0.9999999999999999 0.9999999999999999
  0.9999999999999999]
 [1.0 1.0 1.0 1.0 1.0 0.9 1.0 1.0 0.7 1.0 0.9 1.0 0.9 1.0 1.0 1.0 1.0 1.0
  0.8 1.0 0.9]]

In [50]:
# Sanity check
assert rater_matrix_ta.shape == (8, 21)
assert rater_matrix_cn.shape == (8, 21)

## Compute coefficient

In [51]:
rater_matrix_ta = rater_matrix_ta.astype(float)
rater_matrix_cn = rater_matrix_cn.astype(float)

In [52]:
# Human alpha
alpha_humans_ta = krippendorff.alpha(reliability_data=rater_matrix_ta[:7], level_of_measurement='interval')
alpha_humans_cn = krippendorff.alpha(reliability_data=rater_matrix_cn[:7], level_of_measurement='interval')

In [53]:
# Alpha for all raters (humans + LLM)
alpha_all_ta = krippendorff.alpha(reliability_data=rater_matrix_ta, level_of_measurement='interval')
alpha_all_cn = krippendorff.alpha(reliability_data=rater_matrix_cn, level_of_measurement='interval')

## Report

In [54]:
print(f"Human alpha for TA: {alpha_humans_ta:.3f}",
f"\nHuman alpha for CN: {alpha_humans_cn:.3f}",
f"\nAll raters alpha for TA:{alpha_all_ta:.3f}",
f"\nAll raters alpha for TA:{alpha_all_cn:.3f}",
f'\nDifference TA: {alpha_all_ta - alpha_humans_ta:.3f}',
f'\nDifference CN: {alpha_all_cn - alpha_humans_cn:.3f}',
f'\nOverall difference: {(alpha_all_ta + alpha_all_cn) - (alpha_humans_ta + alpha_humans_cn):.3f}')

Human alpha for TA: 0.246 
Human alpha for CN: 0.126 
All raters alpha for TA:0.218 
All raters alpha for TA:0.128 
Difference TA: -0.028 
Difference CN: 0.002 
Overall difference: -0.027

Inter-rater agreement among humans was low (aprox. 0.13–0.25), indicating that Tone Attunement and Cultural Neutrality are highly subjective constructs in the humanitarian context. Adding the LLM-based metric as an 8th rater did not meaningfully reduce group agreement, suggesting the metric's disagreement profile is comparable to that of the human annotators.

### In-depth analysis

This section analises for which QA pairs did humans agreed/ disagreed strongly.

In [55]:
ta_variance_per_row = rater_matrix_ta[:7].var(axis=0)  # shape (21,)
cn_variance_per_row = rater_matrix_cn[:7].var(axis=0)

In [56]:
ta_variance_per_row

array([0.03890306, 0.05293367, 0.03316327, 0.1065051 , 0.08227041,
       0.03125   , 0.01721939, 0.00637755, 0.08035714, 0.0255102 ,
       0.05548469, 0.01977041, 0.03316327, 0.03571429, 0.00765306,
       0.07334184, 0.0255102 , 0.02104592, 0.09693878, 0.03762755,
       0.0127551 ])

In [57]:
cn_variance_per_row

array([0.02186224, 0.02204082, 0.03132653, 0.04211735, 0.04303571,
       0.06954082, 0.04640306, 0.06586735, 0.05479592, 0.01882653,
       0.05104592, 0.00785714, 0.08454082, 0.03872449, 0.00283163,
       0.00479592, 0.00382653, 0.02571429, 0.13668367, 0.02571429,
       0.02678571])

In [58]:
# Find quartile boundaries
ta_q1, ta_q3 = np.percentile(ta_variance_per_row, [25, 75])
cn_q1, cn_q3 = np.percentile(cn_variance_per_row, [25, 75])

# Indices of low-variance (high-agreement) and high-variance (low-agreement) rows
ta_agree_idx = np.where(ta_variance_per_row <= ta_q1)[0]
ta_disagree_idx = np.where(ta_variance_per_row >= ta_q3)[0]
cn_agree_idx = np.where(cn_variance_per_row <= cn_q1)[0]
cn_disagree_idx = np.where(cn_variance_per_row >= cn_q3)[0]

In [59]:
# For TA: how far is the metric from human consensus on agreed vs contested rows?
human_mean_ta = rater_matrix_ta[:7].mean(axis=0)  # shape (21,)
gap_ta = np.abs(rater_matrix_ta[7] - human_mean_ta)  # absolute distance per row

print(f"TA mean gap (high human agreement rows): {gap_ta[ta_agree_idx].mean():.3f}")
print(f"TA mean gap (low human agreement rows): {gap_ta[ta_disagree_idx].mean():.3f}")

TA mean gap (high human agreement rows): 0.212

TA mean gap (low human agreement rows): 0.231

The gap is essentially the same in both subsets. This means that the metric is systematically offset from human judgement by about 0.2 units, and this offset doesn't depend on whether humans themselves agreed. 

In other words:

When humans agree the answer should score X, the metric scores aprox 0.2 units higher or lower.

When humans disagree among themselves (some score X, some Y), the metric also scores about 0.2 away from their mean.

In [60]:
# Quick check on direction of the offset
signed_gap_ta = rater_matrix_ta[7] - human_mean_ta  # not absolute
print(f"Mean signed gap TA: {signed_gap_ta.mean():.3f}")
print(f"Sign: {'metric > human' if signed_gap_ta.mean() > 0 else 'metric < human'}")

Mean signed gap TA: 0.199

Sign: metric > human

TA is consistently more lenient than humans, confirming the earlier hypothesis.

In [61]:
# For CN: how far is the metric from human consensus on agreed vs contested rows?
human_mean_cn = rater_matrix_cn[:7].mean(axis=0)  # shape (21,)
gap_cn = np.abs(rater_matrix_cn[7] - human_mean_cn)  # absolute distance per row

print(f"CN mean gap (high human agreement rows): {gap_cn[cn_agree_idx].mean():.3f}")
print(f"CN mean gap (low human agreement rows): {gap_cn[cn_disagree_idx].mean():.3f}")

CN mean gap (high human agreement rows): 0.071

CN mean gap (low human agreement rows): 0.191

In [62]:
# Quick check on direction of the offset
signed_gap_cn = rater_matrix_cn[7] - human_mean_cn  # not absolute
print(f"Mean signed gap CN: {signed_gap_cn.mean():.3f}")
print(f"Sign: {'metric > human' if signed_gap_cn.mean() > 0 else 'metric < human'}")

Mean signed gap CN: 0.147

Sign: metric > human

Once again, the metric is more lenient than humans. However, unlike TA, the metric performs well when the task has a clear answer and degrades when the task is contested. 

## Conclusion
The `threshold.ipynb` revealed a potential hypothesis of the metrics being more lenient in judgement compared to humans, especially the TA metric. This notebook proved the following:
- TA: systematically miscalibrated. Off by approx. 0.2 across the board, including on rows where humans were unanimous. Threshold calibration patches this but the underlying metric doesn't track human judgement at row level.
- CN: structurally aligned with human judgement on clear cases, drifts on contested ones. The 3x gap difference between agreement and disagreement subsets is the key signal here.